# Phase 6 — Manual Error Analysis và Paper Synthesis

Notebook này đọc output từ Phase 5 và tạo các bảng gần với báo cáo/paper: benchmark summary, domain robustness, error profile, taxonomy nếu đã annotate, case studies và discussion points.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', 160)

In [6]:
def read_required_csv(path: str) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Không tìm thấy file bắt buộc: {path}')
    df = pd.read_csv(path, encoding='utf-8-sig')
    df.columns = [str(col).replace('\ufeff', '').strip() for col in df.columns]
    return df


def read_optional_csv(candidates: list[str]) -> tuple[pd.DataFrame | None, Path | None]:
    candidates = [Path(path) for path in candidates]
    for path in candidates:
        if path.exists():
            return read_required_csv(path), path
    return None, None

# OUTPUT PHASE 5
PHASE5_DIR = Path("/content/PHASE5")
random_leaderboard = read_required_csv(PHASE5_DIR / 'random_split_leaderboard.csv')
kfold_summary = read_required_csv(PHASE5_DIR / 'kfold_summary.csv')
domain_drop_summary = read_required_csv(PHASE5_DIR / 'domain_drop_summary.csv')
source_robustness_summary = read_required_csv(PHASE5_DIR / 'source_robustness_summary.csv')
category_robustness_summary = read_required_csv(PHASE5_DIR / 'category_robustness_summary.csv')
best_error_table = read_required_csv(PHASE5_DIR / 'best_model_error_table.csv')
error_profile_by_source = read_required_csv(PHASE5_DIR / 'error_profile_by_source.csv')
error_profile_by_category = read_required_csv(PHASE5_DIR / 'error_profile_by_category.csv')
error_profile_by_surface = read_required_csv(PHASE5_DIR / 'error_profile_by_surface_features.csv')
feature_cue_summary = read_required_csv(PHASE5_DIR / 'feature_cue_summary.csv')
case_study_candidates = read_required_csv(PHASE5_DIR / 'case_study_candidates.csv')

# PHASE 6
PHASE6_DIR = Path('/content/PHASE6')
annotation_df, annotation_path = read_optional_csv([
    PHASE5_DIR / 'manual_error_annotation_completed.csv',
])
if annotation_df is None:
    raise FileNotFoundError('Không tìm thấy manual error annotation file hoặc template.')

numeric_tables = [
    random_leaderboard, kfold_summary, domain_drop_summary,
    source_robustness_summary, category_robustness_summary,
    best_error_table, error_profile_by_source, error_profile_by_category,
    error_profile_by_surface, feature_cue_summary, case_study_candidates,
    annotation_df,
]
for df in numeric_tables:
    for col in df.columns:
        if col.endswith('_f1') or col.endswith('_f1_mean') or col.endswith('_f1_std') or col in {
            'accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1',
            'clickbait_precision', 'clickbait_recall', 'roc_auc', 'pr_auc',
            'macro_f1_drop', 'clickbait_f1_drop', 'balanced_accuracy_drop',
            'confidence_in_prediction', 'y_score', 'coefficient', 'abs_coefficient',
        }:
            df[col] = pd.to_numeric(df[col], errors='coerce')

print('Loaded Phase 5 outputs:')
print('random_leaderboard:', random_leaderboard.shape)
print('kfold_summary:', kfold_summary.shape)
print('domain_drop_summary:', domain_drop_summary.shape)
print('best_error_table:', best_error_table.shape)
print('annotation_source:', annotation_path)
print('annotation_df:', annotation_df.shape)

display(random_leaderboard.head())
display(annotation_df.head())

Loaded Phase 5 outputs:
random_leaderboard: (12, 14)
kfold_summary: (8, 20)
domain_drop_summary: (40, 30)
best_error_table: (683, 33)
annotation_source: /content/PHASE5/manual_error_annotation_completed.csv
annotation_df: (100, 14)


,rank_macro_f1,model_name,feature_set,train_size,eval_size,accuracy,balanced_accuracy,macro_f1,weighted_f1,clickbait_precision,clickbait_recall,clickbait_f1,roc_auc,pr_auc
0,1,tfidf_word_logreg,text_title,2389,683,0.751098,0.742134,0.725923,0.757179,0.581749,0.718310,0.642857,0.814954,0.656721
1,2,tfidf_word_char_logreg,text_title,2389,683,0.749634,0.730801,0.720058,0.754297,0.584677,0.680751,0.629067,0.819079,0.666477
2,3,tfidf_word_svm,text_title,2389,683,0.754026,0.722440,0.718378,0.756080,0.599119,0.638498,0.618182,0.802467,0.639669
3,4,tfidf_word_char_svm,text_title,2389,683,0.756955,0.718150,0.717589,0.757263,0.609302,0.615023,0.612150,0.803866,0.649234
4,5,tfidf_char_svm,text_title,2389,683,0.749634,0.723100,0.716338,0.752907,0.588983,0.652582,0.619154,0.801878,0.649820


,id,title,lead_paragraph,source,category,y_true,y_pred,y_score,confidence_in_prediction,error_type,suggested_priority,taxonomy_label,annotator_note,is_case_study_candidate
0,article_0346,Nước thải của Khu du lịch Thiên Cầm 'tra tấn' người dân,"Suốt nhiều tháng qua, người dân ở tổ dân phố Trần Phú (TT.Thiên Cầm, H.Cẩm Xuyên, Hà Tĩnh) bị 'tra tấn' bởi mùi hôi từ nước thải của Khu du lịch Thiên Cầm, ...",Thanh Niên,Other,1,0,0.249300,0.750700,FN,1,FN_neutral_surface_form,"Tiêu đề có sắc thái gây chú ý qua từ 'tra tấn' nhưng vẫn giống tin phản ánh địa phương, nên model coi như non-clickbait.",0
1,article_1074,Mở lối ký ức bằng tư duy hôm nay - Báo Văn hóa,"Giữa những nếp nhà xưa cũ và khung cảnh bình dị của làng Lai Xá, nơi từng vang bóng với nghề ảnh truyền thống, có một bảo tàng nhỏ lặng lẽ kể câu chuyện làn...",Báo Mới,Văn hóa & Nghệ thuật,1,0,0.253598,0.746402,FN,2,FN_domain_specific_clickbait,"Tiêu đề văn hóa có lối viết ẩn dụ và giàu chất chuyên đề, cue clickbait không giống các mẫu tin giật gân thông thường.",0
2,article_2245,"Đánh mẹ, tấn công lực lượng an ninh cơ sở khi được can ngăn","Cà MauBùi Minh Lợi, 30 tuổi, dùng dao đâm trọng thương tổ phó bảo vệ an ninh cơ sở thị trấn Nam Căn khi lực lượng này đến can ngăn anh ta đánh mẹ.",VnExpress,Other,1,0,0.258837,0.741163,FN,3,AMB_borderline_label,Tiêu đề nêu khá đầy đủ sự kiện phạm pháp nên ranh giới clickbait/non-clickbait không thật rõ.,0
3,article_3617,Liên danh Đèo Cả - Văn Phú đề xuất Hà Nội cho kiến tạo 'kỳ tích sông Hồng' - Báo Điện tử Tiếng nói Việt Nam,Liên danh Đèo Cả - Văn Phú vừa được UBND TP. Hà Nội chấp thuận nghiên cứu đề xuất dự án 'Xây dựng Đại lộ và cảnh quan ven sông Hồng' theo phương thức đối tá...,Báo Mới,Other,1,0,0.262077,0.737923,FN,4,FN_subtle_curiosity_gap,Cụm 'kỳ tích sông Hồng' tạo kỳ vọng/curiosity nhưng nằm trong bối cảnh dự án chính sách nên model bỏ sót.,0
4,article_1190,Khi Gen Z dùng công nghệ lập trình nhịp sống số,"Gen Z không chỉ tiếp cận công nghệ mà còn thiết lập cách sống số của họ. Từ AI đến no-code, thế hệ này đang tạo ra những hệ sinh thái cá nhân tự động hóa.",Tuổi Trẻ,Công nghệ & Khoa học,1,0,0.265132,0.734868,FN,5,FN_neutral_surface_form,"Tiêu đề giống một bài phân tích xu hướng công nghệ, bề mặt trung tính nên model không nhận ra nhãn clickbait.",0


In [7]:
# =================== ANNOTATION AUDIT ===================
def is_filled(series: pd.Series) -> pd.Series:
    return series.fillna('').astype(str).str.strip().ne('')


def parse_case_flag(series: pd.Series) -> pd.Series:
    values = series.fillna('').astype(str).str.strip().str.lower()
    return values.isin({'1', 'true', 'yes', 'y', 'x', 'case', 'selected'})

required_annotation_cols = ['id', 'title', 'source', 'category', 'error_type', 'taxonomy_label', 'annotator_note', 'is_case_study_candidate']
missing_annotation_cols = [col for col in required_annotation_cols if col not in annotation_df.columns]

annotation_df = annotation_df.copy()
for col in ['taxonomy_label', 'annotator_note', 'is_case_study_candidate']:
    if col not in annotation_df.columns:
        annotation_df[col] = ''

annotation_df['taxonomy_filled'] = is_filled(annotation_df['taxonomy_label'])
annotation_df['note_filled'] = is_filled(annotation_df['annotator_note'])
annotation_df['case_flag'] = parse_case_flag(annotation_df['is_case_study_candidate'])

n_rows = len(annotation_df)
n_taxonomy = int(annotation_df['taxonomy_filled'].sum())
n_notes = int(annotation_df['note_filled'].sum())
n_case_flags = int(annotation_df['case_flag'].sum())
completion_rate = n_taxonomy / n_rows if n_rows else 0.0
annotation_status = 'ready_for_taxonomy_analysis' if completion_rate >= 0.8 else 'needs_manual_annotation'

annotation_audit = {
    'annotation_source': str(annotation_path),
    'rows': int(n_rows),
    'missing_annotation_columns': missing_annotation_cols,
    'error_type_counts': {str(k): int(v) for k, v in annotation_df['error_type'].value_counts(dropna=False).items()},
    'taxonomy_filled_rows': n_taxonomy,
    'annotator_note_filled_rows': n_notes,
    'case_study_flagged_rows': n_case_flags,
    'taxonomy_completion_rate': completion_rate,
    'annotation_status': annotation_status,
}

with open(PHASE6_DIR / 'phase6_annotation_audit.json', 'w', encoding='utf-8') as f:
    json.dump(annotation_audit, f, ensure_ascii=False, indent=2)

print(json.dumps(annotation_audit, ensure_ascii=False, indent=2))

{
  "annotation_source": "/content/PHASE5/manual_error_annotation_completed.csv",
  "rows": 100,
  "missing_annotation_columns": [],
  "error_type_counts": {
    "FN": 50,
    "FP": 50
  },
  "taxonomy_filled_rows": 100,
  "annotator_note_filled_rows": 100,
  "case_study_flagged_rows": 15,
  "taxonomy_completion_rate": 1.0,
  "annotation_status": "ready_for_taxonomy_analysis"
}


In [8]:
# ====================== PAPER-READY BENCHMARK TABLES ======================
def round_metric_table(df: pd.DataFrame, decimals: int = 4) -> pd.DataFrame:
    out = df.copy()
    numeric_cols = out.select_dtypes(include='number').columns
    out[numeric_cols] = out[numeric_cols].round(decimals)
    return out

paper_table_random_benchmark = random_leaderboard[[
    'rank_macro_f1', 'model_name', 'macro_f1', 'clickbait_precision',
    'clickbait_recall', 'clickbait_f1', 'roc_auc', 'pr_auc', 'balanced_accuracy',
]].copy()
paper_table_random_benchmark = round_metric_table(paper_table_random_benchmark)

paper_table_kfold_summary = kfold_summary[[
    'rank_macro_f1_mean', 'model_name', 'macro_f1_mean', 'macro_f1_std',
    'clickbait_f1_mean', 'clickbait_f1_std', 'balanced_accuracy_mean', 'balanced_accuracy_std',
]].copy()
paper_table_kfold_summary = round_metric_table(paper_table_kfold_summary)

paper_table_random_benchmark.to_csv(PHASE6_DIR / 'paper_table_random_benchmark.csv', index=False)
paper_table_kfold_summary.to_csv(PHASE6_DIR / 'paper_table_kfold_summary.csv', index=False)

display(paper_table_random_benchmark)
display(paper_table_kfold_summary)

,rank_macro_f1,model_name,macro_f1,clickbait_precision,clickbait_recall,clickbait_f1,roc_auc,pr_auc,balanced_accuracy
0,1,tfidf_word_logreg,0.7259,0.5817,0.7183,0.6429,0.8150,0.6567,0.7421
1,2,tfidf_word_char_logreg,0.7201,0.5847,0.6808,0.6291,0.8191,0.6665,0.7308
2,3,tfidf_word_svm,0.7184,0.5991,0.6385,0.6182,0.8025,0.6397,0.7224
3,4,tfidf_word_char_svm,0.7176,0.6093,0.6150,0.6121,0.8039,0.6492,0.7182
4,5,tfidf_char_svm,0.7163,0.5890,0.6526,0.6192,0.8019,0.6498,0.7231
5,6,tfidf_char_logreg,0.7132,0.5698,0.6901,0.6242,0.8117,0.6564,0.7270
6,7,tfidf_word_rf,0.6878,0.6513,0.4648,0.5425,0.8127,0.6545,0.6760
7,8,keyword_heuristic_threshold_1.00,0.6711,0.6855,0.3991,0.5045,0.6611,0.4767,0.6580
8,9,tfidf_word_nb,0.6068,0.7681,0.2488,0.3759,0.8063,0.6411,0.6074
9,10,random_stratified,0.5079,0.3226,0.3286,0.3256,NaN,NaN,0.5079


,rank_macro_f1_mean,model_name,macro_f1_mean,macro_f1_std,clickbait_f1_mean,clickbait_f1_std,balanced_accuracy_mean,balanced_accuracy_std
0,1,tfidf_word_char_logreg,0.7464,0.0048,0.6601,0.0076,0.7545,0.0062
1,2,tfidf_word_logreg,0.7452,0.0103,0.6626,0.0141,0.7572,0.0113
2,3,tfidf_word_char_svm,0.7422,0.0224,0.6467,0.0286,0.7432,0.0214
3,4,tfidf_char_logreg,0.7404,0.0128,0.6561,0.0140,0.7519,0.0106
4,5,tfidf_word_svm,0.7401,0.0171,0.6446,0.0269,0.7423,0.0201
5,6,tfidf_char_svm,0.7359,0.0213,0.6400,0.0282,0.7386,0.0212
6,7,tfidf_word_nb,0.6234,0.0269,0.4043,0.0447,0.6199,0.0203
7,8,majority_class,0.4076,0.0001,0.0000,0.0000,0.5000,0.0000


In [9]:
# ========================= PAPER READY DOMAIN ROBUSTNESS TABLE =========================
domain_cols = [
    'domain_type', 'domain_name', 'model_name', 'macro_f1', 'clickbait_f1',
    'macro_f1_drop', 'clickbait_f1_drop', 'balanced_accuracy', 'balanced_accuracy_drop',
]

hard_by_low_macro = domain_drop_summary.sort_values(['macro_f1', 'clickbait_f1'], ascending=True).head(12)
hard_by_macro_drop = domain_drop_summary.sort_values(['macro_f1_drop', 'clickbait_f1_drop'], ascending=False).head(12)
hard_by_clickbait_drop = domain_drop_summary.sort_values(['clickbait_f1_drop', 'macro_f1_drop'], ascending=False).head(12)

paper_table_domain_robustness = pd.concat([
    hard_by_low_macro.assign(selection_reason='lowest_macro_f1'),
    hard_by_macro_drop.assign(selection_reason='largest_macro_f1_drop'),
    hard_by_clickbait_drop.assign(selection_reason='largest_clickbait_f1_drop'),
], ignore_index=True)
paper_table_domain_robustness = paper_table_domain_robustness.drop_duplicates(
    subset=['domain_type', 'domain_name', 'model_name'],
    keep='first',
)
keep_cols = ['selection_reason'] + domain_cols
paper_table_domain_robustness = paper_table_domain_robustness[keep_cols]
paper_table_domain_robustness = round_metric_table(paper_table_domain_robustness)
paper_table_domain_robustness.to_csv(PHASE6_DIR / 'paper_table_domain_robustness.csv', index=False)

display(paper_table_domain_robustness)

,selection_reason,domain_type,domain_name,model_name,macro_f1,clickbait_f1,macro_f1_drop,clickbait_f1_drop,balanced_accuracy,balanced_accuracy_drop
0,lowest_macro_f1,category,Giải trí & Showbiz,tfidf_word_svm,0.5708,0.6573,0.1476,-0.0392,0.6083,0.1141
1,lowest_macro_f1,category,Quốc tế,tfidf_word_svm,0.5868,0.4342,0.1316,0.1840,0.6120,0.1104
2,lowest_macro_f1,category,Giải trí & Showbiz,tfidf_word_logreg,0.5917,0.7101,0.1342,-0.0672,0.6101,0.1320
3,lowest_macro_f1,category,Quốc tế,tfidf_word_logreg,0.6007,0.4366,0.1253,0.2062,0.6174,0.1247
4,lowest_macro_f1,category,Tin tức tổng hợp,tfidf_word_logreg,0.6062,0.3043,0.1197,0.3385,0.8555,-0.1134
5,lowest_macro_f1,category,Tin tức tổng hợp,tfidf_word_svm,0.6111,0.3111,0.1072,0.3071,0.8581,-0.1357
6,lowest_macro_f1,category,Thể thao,tfidf_word_logreg,0.6183,0.4574,0.1076,0.1854,0.6294,0.1127
7,lowest_macro_f1,source,VnExpress,tfidf_word_logreg,0.6288,0.4444,0.0971,0.1984,0.7421,0.0000
8,lowest_macro_f1,category,Thể thao,tfidf_word_svm,0.6389,0.5000,0.0795,0.1182,0.6607,0.0618
9,lowest_macro_f1,source,VnExpress,tfidf_word_svm,0.6409,0.4460,0.0774,0.1721,0.7292,-0.0068


In [10]:
# ========================== PAPER-READY ERROR PROFILE TABLE ==========================
def prepare_error_profile(profile_df: pd.DataFrame, group_type: str, group_col: str) -> pd.DataFrame:
    out = profile_df.copy()
    out.insert(0, 'group_type', group_type)
    out = out.rename(columns={group_col: 'group_name'})
    keep = [
        'group_type', 'group_name', 'n_samples', 'n_clickbait', 'n_non_clickbait',
        'macro_f1', 'clickbait_f1', 'false_positive_count', 'false_negative_count',
        'false_positive_rate', 'false_negative_rate', 'dominant_error_type',
    ]
    return out[[col for col in keep if col in out.columns]]

source_profile = prepare_error_profile(error_profile_by_source, 'source', 'source')
category_profile = prepare_error_profile(error_profile_by_category, 'category', 'category')
surface_profile = error_profile_by_surface.copy().rename(columns={'group_value': 'group_name'})
surface_profile.insert(0, 'group_type', surface_profile['feature_group'])
surface_profile = surface_profile[[
    'group_type', 'group_name', 'n_samples', 'n_clickbait', 'n_non_clickbait',
    'macro_f1', 'clickbait_f1', 'false_positive_count', 'false_negative_count',
    'false_positive_rate', 'false_negative_rate', 'dominant_error_type',
]]

paper_table_error_profile = pd.concat([source_profile, category_profile, surface_profile], ignore_index=True)
paper_table_error_profile = paper_table_error_profile.sort_values(['macro_f1', 'clickbait_f1'], ascending=True).reset_index(drop=True)
paper_table_error_profile = round_metric_table(paper_table_error_profile)
paper_table_error_profile.to_csv(PHASE6_DIR / 'paper_table_error_profile.csv', index=False)

display(paper_table_error_profile.head(30))

,group_type,group_name,n_samples,n_clickbait,n_non_clickbait,macro_f1,clickbait_f1,false_positive_count,false_negative_count,false_positive_rate,false_negative_rate,dominant_error_type
0,source,Kênh14,9,5,4,0.5500,0.6000,2,2,0.5000,0.4000,FN
1,length_bin,very_short,68,28,40,0.5850,0.5484,17,11,0.4250,0.3929,FP
2,category,Giải trí & Showbiz,51,35,16,0.6029,0.7353,8,10,0.5000,0.2857,FN
3,category,Văn hóa & Nghệ thuật,11,4,7,0.6071,0.5000,2,2,0.2857,0.5000,FN
4,category,Thể thao,64,12,52,0.6135,0.4444,16,4,0.3077,0.3333,FP
5,category,Chính trị & An ninh,26,6,20,0.6286,0.4000,2,4,0.1000,0.6667,FN
6,has_question,has_question,55,41,14,0.6319,0.8193,8,7,0.5714,0.1707,FP
7,source,VnExpress,131,20,111,0.6343,0.4444,29,6,0.2613,0.3000,FP
8,category,Du lịch & Ẩm thực,23,13,10,0.6349,0.7143,5,3,0.5000,0.2308,FP
9,category,Sức khỏe & Đời sống,40,19,21,0.6419,0.6957,11,3,0.5238,0.1579,FP


In [11]:
# ====================== PAPER-READY FEATURE CUE TABLE ======================
top_clickbait = feature_cue_summary[feature_cue_summary['direction'] == 'clickbait'].head(20).copy()
top_non_clickbait = feature_cue_summary[feature_cue_summary['direction'] == 'non_clickbait'].head(20).copy()
paper_table_feature_cues = pd.concat([top_clickbait, top_non_clickbait], ignore_index=True)
paper_table_feature_cues = paper_table_feature_cues[[
    'direction', 'rank_within_direction', 'feature', 'coefficient', 'cue_group',
]]
paper_table_feature_cues = round_metric_table(paper_table_feature_cues)
paper_table_feature_cues.to_csv(PHASE6_DIR / 'paper_table_feature_cues.csv', index=False)

display(paper_table_feature_cues)

,direction,rank_within_direction,feature,coefficient,cue_group
0,clickbait,1,nào,2.1010,question/curiosity
1,clickbait,2,sao,2.0012,question/curiosity
2,clickbait,3,có,1.8794,vague_reference
3,clickbait,4,bất,1.8512,surprise/emotion
4,clickbait,5,gì,1.8374,question/curiosity
5,clickbait,6,ngờ,1.7100,surprise/emotion
6,clickbait,7,không,1.6802,vague_reference
7,clickbait,8,con,1.6250,other
8,clickbait,9,ăn,1.5770,other
9,clickbait,10,khiến,1.5047,vague_reference


In [12]:
# =========================== ERROR TAXONOMY TABLE =========================
def make_taxonomy_table(annotation_df: pd.DataFrame) -> pd.DataFrame:
    annotated = annotation_df[annotation_df['taxonomy_filled']].copy()
    if annotated.empty:
        return pd.DataFrame(columns=[
            'error_type', 'taxonomy_label', 'count', 'percentage_within_error_type', 'percentage_overall',
        ])

    total = len(annotated)
    rows = []
    grouped = annotated.groupby(['error_type', 'taxonomy_label'], dropna=False).size().reset_index(name='count')
    error_type_totals = annotated.groupby('error_type').size().to_dict()
    for _, row in grouped.iterrows():
        error_type = row['error_type']
        count = int(row['count'])
        rows.append({
            'error_type': error_type,
            'taxonomy_label': row['taxonomy_label'],
            'count': count,
            'percentage_within_error_type': count / error_type_totals.get(error_type, count),
            'percentage_overall': count / total,
        })
    out = pd.DataFrame(rows)
    return out.sort_values(['error_type', 'count'], ascending=[True, False]).reset_index(drop=True)

paper_table_error_taxonomy = make_taxonomy_table(annotation_df)
paper_table_error_taxonomy = round_metric_table(paper_table_error_taxonomy)
paper_table_error_taxonomy.to_csv(PHASE6_DIR / 'paper_table_error_taxonomy.csv', index=False)

display(paper_table_error_taxonomy)

,error_type,taxonomy_label,count,percentage_within_error_type,percentage_overall
0,FN,FN_subtle_curiosity_gap,18,0.36,0.18
1,FN,FN_domain_specific_clickbait,10,0.20,0.10
2,FN,FN_neutral_surface_form,7,0.14,0.07
3,FN,AMB_borderline_label,6,0.12,0.06
4,FN,FN_entity_context_needed,4,0.08,0.04
5,FN,DATA_quality_issue,3,0.06,0.03
6,FN,FN_requires_lead_context,2,0.04,0.02
7,FP,FP_emotional_but_informative,21,0.42,0.21
8,FP,FP_entertainment_style_bias,13,0.26,0.13
9,FP,FP_question_but_legitimate,8,0.16,0.08


In [13]:
# ================================== CASE STUDIES ==================================
def select_case_studies(annotation_df: pd.DataFrame, fallback_df: pd.DataFrame, n: int = 12) -> pd.DataFrame:
    flagged = annotation_df[annotation_df['case_flag']].copy()
    if not flagged.empty:
        flagged['case_source'] = 'manual_flag'
        selected = flagged
        max_rows = len(flagged)
    elif annotation_status == 'ready_for_taxonomy_analysis':
        selected = annotation_df.sort_values('confidence_in_prediction', ascending=False).head(n).copy()
        selected['case_source'] = 'annotated_high_confidence'
        max_rows = n
    else:
        selected = fallback_df.copy()
        selected['case_source'] = 'phase5_candidate_fallback'
        max_rows = n

    for col in ['taxonomy_label', 'annotator_note', 'analysis_note']:
        if col not in selected.columns:
            selected[col] = ''

    keep_cols = [
        'case_source', 'id', 'title', 'lead_paragraph', 'source', 'category',
        'y_true', 'y_pred', 'error_type', 'y_score', 'confidence_in_prediction',
        'taxonomy_label', 'annotator_note', 'analysis_note',
    ]
    keep_cols = [col for col in keep_cols if col in selected.columns]
    return selected[keep_cols].head(max_rows).reset_index(drop=True)

paper_case_studies = select_case_studies(annotation_df, case_study_candidates, n=12)
paper_case_studies.to_csv(PHASE6_DIR / 'paper_case_studies.csv', index=False)

display(paper_case_studies)


,case_source,id,title,lead_paragraph,source,category,y_true,y_pred,error_type,y_score,confidence_in_prediction,taxonomy_label,annotator_note,analysis_note
0,manual_flag,article_3648,Cảng cá 280 tỷ đồng sắp hoạt động nhưng luồng lạch bồi lắng,"Dự án cảng cá Cửa Nhượng (Hà Tĩnh) có mức đầu tư 280 tỷ đồng được đánh giá hiện đại bậc nhất miền Trung đã hoàn thiện, chuẩn bị đưa vào hoạt động song luồng...",24h,Giải trí & Showbiz,1,0,FN,0.327111,0.672889,DATA_quality_issue,"Category là Giải trí & Showbiz nhưng nội dung là cảng cá/hạ tầng, cho thấy metadata có thể bị lệch.",
1,manual_flag,article_2131,"Video: Nam sinh xem điểm thi vào lớp 10, phản ứng khiến cả cõi mạng vui lây","Giữa mùa công bố điểm thi vào lớp 10 đầy hồi hộp, mạng xã hội lại rộn ràng vì đoạn clip ghi lại phản ứng hài hước của một nam sinh khi biết mình đỗ cấp 3",SaoStar,Giáo dục & Học đường,1,0,FN,0.355583,0.644417,FN_subtle_curiosity_gap,Cụm 'phản ứng khiến cả cõi mạng vui lây' che giấu phản ứng cụ thể và là ví dụ clickbait rõ.,
2,manual_flag,article_3408,"Cao nguyên cách Hà Nội 300km có biển mây, thác nước, khách rộn ràng reo vui","Quãng đường di chuyển thuận tiện với tuyến cao tốc Hà Nội - Lào Cai, không khí trong lành, mát mẻ, mùa hè chỉ khoảng 20 độ C... là những lý do khiến cao ngu...",VietNamNet,Du lịch & Ẩm thực,1,0,FN,0.415434,0.584566,FN_subtle_curiosity_gap,Tiêu đề mô tả địa điểm bằng khoảng cách và trải nghiệm nhưng không nêu tên nơi ngay ở headline.,
3,manual_flag,article_2870,Top đội bóng,NaN,Thanh Niên,Chính trị & An ninh,1,0,FN,0.415857,0.584143,DATA_quality_issue,"Tiêu đề quá ngắn và lead trống, không đủ ngữ cảnh để đánh giá nhãn một cách đáng tin cậy.",
4,manual_flag,article_2031,Khám phá,"Khám phá du lịch - Khám phá tour du lịch độc đáo, nổi tiếng. Khám phá văn hóa ẩm thực vùng miền, địa điểm, địa danh nổi tiếng Việt Nam và thế giới",Thanh Niên,Chính trị & An ninh,1,0,FN,0.419430,0.580570,DATA_quality_issue,"Tiêu đề chỉ là một từ chung chung và lead giống mô tả chuyên mục, khả năng là lỗi scrape/metadata.",
5,manual_flag,article_2732,Tắt các thiết lập này sẽ giúp tăng tốc Windows 11 nhanh hơn rất nhiều - Báo Pháp Luật TP.HCM,"Windows 11 mang đến giao diện hiện đại, nhiều hiệu ứng chuyển động mượt mà, nhưng chính những yếu tố này lại khiến hệ thống trở nên ì ạch. Làm thế nào để tă...",Báo Mới,Other,1,0,FN,0.425454,0.574546,FN_subtle_curiosity_gap,"Cụm 'các thiết lập này' che giấu thông tin cụ thể, là curiosity gap dạng hướng dẫn.",
6,manual_flag,article_0718,"Người đàn ông đến ngân hàng chuyển 22 triệu đồng cho vợ tương lai, nhân viên liền báo cảnh sát",Người đàn ông phản ứng vô cùng gay gắt khi làm viêc với phía cảnh sát.,SaoStar,Quốc tế,1,0,FN,0.440217,0.559783,FN_subtle_curiosity_gap,"Tiêu đề kể tình huống lạ nhưng giấu lý do nhân viên báo cảnh sát, curiosity gap rất rõ.",
7,manual_flag,article_1226,Mặt trời giả xuất hiện ở Việt Nam đúng ngày hạ chí: Báo hiệu điều gì?,"Hạ chí 21.6 hôm qua, nhiều người bất ngờ nhìn thấy hiện tượng mặt trời giả từ núi Bà Đen (Tây Ninh). Nguyên nhân của hiện tượng này là gì?",Thanh Niên,Other,1,0,FN,0.449181,0.550819,FN_subtle_curiosity_gap,Câu hỏi 'Báo hiệu điều gì?' che giấu lời giải thích và là cue curiosity điển hình.,
8,manual_flag,article_1237,"Tưởng bị phạt nguội, tài xế vượt đèn đỏ nhận thông báo không ngờ từ CSGT","Tài xế vượt đèn đỏ đưa người bị tai nạn đi cấp cứu ở TP.Đà Lạt lo bị phạt nguội, nhưng thông báo mới nhất từ CSGT khiến ông bất ngờ.",Thanh Niên,Other,1,0,FN,0.454740,0.545260,FN_subtle_curiosity_gap,"Cụm 'thông báo không ngờ' che giấu kết quả, tạo clickbait rõ.",
9,manual_flag,article_1499,Alo bác sĩ nghe: Đau mỏi vai gáy khi ngủ dậy cảnh báo điều gì?,"'Tôi năm nay 55 tuổi, gần đây hay đau mỏi vai gáy, nhất là buổi sáng ngủ dậy. Liệu có phải dấu hiệu của thoái hóa đốt sống cổ không? Tôi nên khám và điều tr...",Thanh Niên,Other,0,1,FP,0.804687,0.804687,FP_question_but_legitimate,"Tiêu đề là câu hỏi tư vấn y tế hợp lệ, không cố tình che giấu thông tin quá mức.",


In [14]:
# =============================== DISCUSSION POINTS ===============================
def fmt(value, digits: int = 4) -> str:
    if pd.isna(value):
        return 'NA'
    return f'{float(value):.{digits}f}'

best_random = random_leaderboard.iloc[0]
best_kfold = kfold_summary.iloc[0]
worst_domain = domain_drop_summary.sort_values(['macro_f1', 'clickbait_f1'], ascending=True).iloc[0]
largest_macro_drop = domain_drop_summary.sort_values('macro_f1_drop', ascending=False).iloc[0]
largest_clickbait_drop = domain_drop_summary.sort_values('clickbait_f1_drop', ascending=False).iloc[0]
error_counts = best_error_table['error_type'].value_counts().to_dict()

fp_count = int(error_counts.get('FP', 0))
fn_count = int(error_counts.get('FN', 0))

if annotation_status == 'ready_for_taxonomy_analysis' and not paper_table_error_taxonomy.empty:
    top_taxonomy = paper_table_error_taxonomy.sort_values(
        ['count', 'percentage_overall'],
        ascending=False,
    ).iloc[0]
    taxonomy_sentence = (
        f"Taxonomy annotation is complete; the most frequent annotated error category overall is "
        f"`{top_taxonomy['taxonomy_label']}` under `{top_taxonomy['error_type']}` "
        f"({int(top_taxonomy['count'])} cases, {fmt(top_taxonomy['percentage_overall'])} overall)."
    )
else:
    taxonomy_sentence = (
        'Manual taxonomy annotation is not complete yet; use `manual_error_annotation_template.csv` '
        'or create `manual_error_annotation_completed.csv` before writing final taxonomy analysis.'
    )

discussion_lines = [
    '# Phase 6 Discussion Points',
    '',
    '## Benchmark',
    '',
    f"- Best random split model: `{best_random['model_name']}` with Macro-F1 = {fmt(best_random['macro_f1'])} and Clickbait F1 = {fmt(best_random['clickbait_f1'])}.",
    f"- Best k-fold model by mean Macro-F1: `{best_kfold['model_name']}` with Macro-F1 mean = {fmt(best_kfold['macro_f1_mean'])} ± {fmt(best_kfold['macro_f1_std'])}.",
    '- Accuracy should remain secondary because the majority baseline has non-trivial accuracy but zero Clickbait F1.',
    '',
    '## Domain Robustness',
    '',
    f"- Hardest held-out domain by Macro-F1: `{worst_domain['domain_type']}` = `{worst_domain['domain_name']}` using `{worst_domain['model_name']}`; Macro-F1 = {fmt(worst_domain['macro_f1'])}.",
    f"- Largest Macro-F1 drop: `{largest_macro_drop['domain_type']}` = `{largest_macro_drop['domain_name']}` using `{largest_macro_drop['model_name']}`; drop = {fmt(largest_macro_drop['macro_f1_drop'])}.",
    f"- Largest Clickbait F1 drop: `{largest_clickbait_drop['domain_type']}` = `{largest_clickbait_drop['domain_name']}` using `{largest_clickbait_drop['model_name']}`; drop = {fmt(largest_clickbait_drop['clickbait_f1_drop'])}.",
    '',
    '## Error Analysis',
    '',
    f"- Best model error counts on random test: `{error_counts}`.",
    f"- The model produces more false positives ({fp_count}) than false negatives ({fn_count}), suggesting a tendency to over-predict clickbait for some informative but attention-grabbing headlines.",
    f"- {taxonomy_sentence}",
    '',
    '## Explainability',
    '',
    '- Logistic Regression coefficients should be described as lexical associations, not causal explanations.',
    '- Top clickbait cues can support analysis of Vietnamese curiosity/question framing, while top non-clickbait cues often reflect official/news discourse.',
]

paper_discussion_points = '\n'.join(discussion_lines) + '\n'
(PHASE6_DIR / 'paper_discussion_points.md').write_text(paper_discussion_points, encoding='utf-8')
print(paper_discussion_points)


# Phase 6 Discussion Points

## Benchmark

- Best random split model: `tfidf_word_logreg` with Macro-F1 = 0.7259 and Clickbait F1 = 0.6429.
- Best k-fold model by mean Macro-F1: `tfidf_word_char_logreg` with Macro-F1 mean = 0.7464 ± 0.0048.
- Accuracy should remain secondary because the majority baseline has non-trivial accuracy but zero Clickbait F1.

## Domain Robustness

- Hardest held-out domain by Macro-F1: `category` = `Giải trí & Showbiz` using `tfidf_word_svm`; Macro-F1 = 0.5708.
- Largest Macro-F1 drop: `category` = `Giải trí & Showbiz` using `tfidf_word_svm`; drop = 0.1476.
- Largest Clickbait F1 drop: `category` = `Tin tức tổng hợp` using `tfidf_word_logreg`; drop = 0.3385.

## Error Analysis

- Best model error counts on random test: `{'TN': 360, 'TP': 153, 'FP': 110, 'FN': 60}`.
- The model produces more false positives (110) than false negatives (60), suggesting a tendency to over-predict clickbait for some informative but attention-grabbing headlines.
- Taxonomy annotati

In [15]:
# ============================ PHASE6 SUMMARY ============================
summary_lines = [
    '# Phase 6 Summary',
    '',
    f"- Phase 5 directory: `{PHASE5_DIR}`",
    f"- Phase 6 directory: `{PHASE6_DIR}`",
    f"- Annotation source: `{annotation_path}`",
    f"- Annotation status: `{annotation_status}`",
    f"- Taxonomy completion rate: {fmt(completion_rate)}",
    f"- Annotated rows: {n_taxonomy}/{n_rows}",
    f"- Case study rows exported: {len(paper_case_studies)}",
    '',
    '## Main outputs',
    '',
    '- `phase6_annotation_audit.json`',
    '- `paper_table_random_benchmark.csv`',
    '- `paper_table_kfold_summary.csv`',
    '- `paper_table_domain_robustness.csv`',
    '- `paper_table_error_profile.csv`',
    '- `paper_table_feature_cues.csv`',
    '- `paper_table_error_taxonomy.csv`',
    '- `paper_case_studies.csv`',
    '- `paper_discussion_points.md`',
]

phase6_summary = '\n'.join(summary_lines) + '\n'
(PHASE6_DIR / 'phase6_summary.md').write_text(phase6_summary, encoding='utf-8')
print(phase6_summary)

# Phase 6 Summary

- Phase 5 directory: `/content/PHASE5`
- Phase 6 directory: `/content/PHASE6`
- Annotation source: `/content/PHASE5/manual_error_annotation_completed.csv`
- Annotation status: `ready_for_taxonomy_analysis`
- Taxonomy completion rate: 1.0000
- Annotated rows: 100/100
- Case study rows exported: 12

## Main outputs

- `phase6_annotation_audit.json`
- `paper_table_random_benchmark.csv`
- `paper_table_kfold_summary.csv`
- `paper_table_domain_robustness.csv`
- `paper_table_error_profile.csv`
- `paper_table_feature_cues.csv`
- `paper_table_error_taxonomy.csv`
- `paper_case_studies.csv`
- `paper_discussion_points.md`



In [17]:
# ====================== SANITY CHECK OUTPUTS ==================================
expected_outputs = [
    'phase6_annotation_audit.json',
    'paper_table_random_benchmark.csv',
    'paper_table_kfold_summary.csv',
    'paper_table_domain_robustness.csv',
    'paper_table_error_profile.csv',
    'paper_table_feature_cues.csv',
    'paper_table_error_taxonomy.csv',
    'paper_case_studies.csv',
    'paper_discussion_points.md',
    'phase6_summary.md',
]

missing_outputs = [name for name in expected_outputs if not (PHASE6_DIR / name).exists()]
print('Missing outputs:', missing_outputs)
print('Exported Phase 6 files:')
for path in sorted(PHASE6_DIR.glob('*')):
    print(f'  - {path}')

if missing_outputs:
    raise FileNotFoundError(f'Missing expected Phase 6 outputs: {missing_outputs}')

Missing outputs: []
Exported Phase 6 files:
  - /content/PHASE6/paper_case_studies.csv
  - /content/PHASE6/paper_discussion_points.md
  - /content/PHASE6/paper_table_domain_robustness.csv
  - /content/PHASE6/paper_table_error_profile.csv
  - /content/PHASE6/paper_table_error_taxonomy.csv
  - /content/PHASE6/paper_table_feature_cues.csv
  - /content/PHASE6/paper_table_kfold_summary.csv
  - /content/PHASE6/paper_table_random_benchmark.csv
  - /content/PHASE6/phase6_annotation_audit.json
  - /content/PHASE6/phase6_summary.md
